# Phase 3 — GraphRAG Retrieval Precision@5 (V1)

**Purpose:** Evaluate whether the top five retrieved evidence records are relevant to each risk investigation. This is an **independent relevance-label benchmark**, not an LLM-generated answer score. No retraining, no API key, no GPU.

**Important:** Precision@5 is **not calculated until actual relevance judgments are entered**. This notebook does not estimate Recall@5 because the complete universe of relevant evidence is not labelled. The sample is stratified/purposive and results are descriptive, not a population estimate.

In [2]:
from pathlib import Path
import json, hashlib
import pandas as pd
import numpy as np
from IPython.display import display
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Outside Colab: set ROOT to a local folder')
ROOT=Path('/content/drive/MyDrive/GC1/Temporal_snapshots')
QUEUE=ROOT/'graph_hybrid_retrieval_event_focused_full_v2_output'/'agentic_graphrag_retrieval_queue.jsonl'
OUT=Path('/content/drive/MyDrive/03_Investigation_Explanation/Retrieval_Evaluation/Results/Precision_at_5_V1')
OUT.mkdir(parents=True,exist_ok=True)
TOP_K=5
SAMPLE_N=30
print('Queue:',QUEUE,'exists:',QUEUE.exists())
assert QUEUE.is_file(), f'Missing retrieval queue: {QUEUE}'


Mounted at /content/drive
Queue: /content/drive/MyDrive/GC1/Temporal_snapshots/graph_hybrid_retrieval_event_focused_full_v2_output/agentic_graphrag_retrieval_queue.jsonl exists: True


## 1. Load and validate the original retrieval queue

This notebook evaluates the order already present in each retrieval response; **the saved retrieval order is retained without citation-based re-sorting frequency or by LLM output**. A row's `rank` is its original position in `evidence`.

In [3]:
queue=[json.loads(line) for line in QUEUE.read_text(encoding='utf-8-sig').splitlines() if line.strip()]
assert len(queue)==len({r['trigger_id'] for r in queue}),'Duplicate triggers'
assert all(isinstance(r.get('evidence'),list) for r in queue)
print('Investigations:',len(queue))
print('Partitions:',pd.Series([r.get('partition') for r in queue]).value_counts().to_dict())
print('Retrieval statuses:',pd.Series([r.get('retrieval_status') for r in queue]).value_counts().to_dict())


Investigations: 139
Partitions: {'validation': 74, 'exploratory_test': 65}
Retrieval statuses: {'READY_FOR_AGENTIC_GRAPHRAG': 125, 'INSUFFICIENT_EVIDENCE': 14}


## 2. Select 30 investigations, without using relevance labels

Deterministic round-robin over risk, partition, and retrieval sufficiency. This includes difficult/insufficient cases and avoids cherry-picking high-quality reports. If your study requires a probabilistic estimate, design a probability sample instead.

In [4]:
def risk_bucket(r):
    types=r.get('risk_types') or []
    return str(types[0]) if types else 'unknown'
ordered=sorted(queue,key=lambda r:(str(r.get('trigger_id'))))
strata={}
for r in ordered:
    key=(str(r.get('partition','unknown')),risk_bucket(r),str(r.get('retrieval_status','unknown')))
    strata.setdefault(key,[]).append(r)
selected=[]; seen=set()
while len(selected)<min(SAMPLE_N,len(queue)):
    changed=False
    for key in sorted(strata):
        while strata[key] and strata[key][0]['trigger_id'] in seen: strata[key].pop(0)
        if strata[key] and len(selected)<SAMPLE_N:
            r=strata[key].pop(0);selected.append(r);seen.add(r['trigger_id']);changed=True
    if not changed:break
assert len(selected)==min(SAMPLE_N,len(queue))
print('Selected:',len(selected))
display(pd.DataFrame([{'partition':r.get('partition'),'risk':risk_bucket(r),'status':r.get('retrieval_status')} for r in selected]).value_counts().rename('cases').reset_index())


Selected: 30


,partition,risk,status,cases
0,exploratory_test,dependency,READY_FOR_AGENTIC_GRAPHRAG,4
1,exploratory_test,schedule,INSUFFICIENT_EVIDENCE,3
2,exploratory_test,schedule,READY_FOR_AGENTIC_GRAPHRAG,3
3,validation,schedule,READY_FOR_AGENTIC_GRAPHRAG,3
4,exploratory_test,scope,READY_FOR_AGENTIC_GRAPHRAG,3
5,validation,schedule,INSUFFICIENT_EVIDENCE,3
6,validation,dependency,READY_FOR_AGENTIC_GRAPHRAG,3
7,validation,scope,READY_FOR_AGENTIC_GRAPHRAG,3
8,exploratory_test,dependency,INSUFFICIENT_EVIDENCE,2
9,validation,dependency,INSUFFICIENT_EVIDENCE,2


## 3. Export the top-five evidence for blinded relevance review

**Evidence-item annotation uses** `1` (relevant), `0` (not relevant), or leave blank if undecided. Relevance is assessed according to whether the item contains substantive information useful for investigating the specified risk **at that checkpoint**, not merely whether it shares project names or appears in the LLM answer. Static metadata without substantive relevance should normally be 0. Do not use the LLM's answer as a reference label.

A second independent reviewer is recommended for a subset. Preserve the first judgments before adjudication.

In [5]:
rows=[]
for r in selected:
    evs=r.get('evidence',[])
    for rank,e in enumerate(evs[:TOP_K],start=1):
        rows.append({'trigger_id':r['trigger_id'],'instance_id':r.get('instance_id'),
          'checkpoint_time':r.get('checkpoint_time'),'partition':r.get('partition'),
          'risk_type':risk_bucket(r),'risk_types':json.dumps(r.get('risk_types',[])),
          'retrieval_status':r.get('retrieval_status'),'investigation_query':r.get('query'),
          'rank':rank,'evidence_id':e.get('evidence_id'),
          'evidence_kind':e.get('evidence_kind'),'available_at':e.get('available_at'),
          'evidence_text':e.get('text'),'source_id':e.get('source_id'),
          'relevant':'','reviewer_notes':'','reviewer_id':''})
review=pd.DataFrame(rows)
assert not review.duplicated(['trigger_id','rank']).any()
assert review['rank'].between(1,TOP_K).all()
review_path=OUT/'retrieval_relevance_review_top5.csv'
if review_path.exists():
    print('Existing review preserved; not overwritten:',review_path)
    template=OUT/'retrieval_relevance_review_top5_TEMPLATE_NEW.csv'
    review.to_csv(template,index=False)
else:
    review.to_csv(review_path,index=False)
print('Evidence rows:',len(review),'across',review.trigger_id.nunique(),'investigations')
display(review[['trigger_id','risk_type','rank','evidence_kind','evidence_text','relevant']].head(10))


Evidence rows: 144 across 30 investigations


,trigger_id,risk_type,rank,evidence_kind,evidence_text,relevant
0,b72e0d20a23d00343eeb,dependency,1,KG_HistoricalIssue,Jira temporal graph HistoricalIssue; source ID...,
1,b72e0d20a23d00343eeb,dependency,2,KG_HistoricalIssue,Jira temporal graph HistoricalIssue; source ID...,
2,b72e0d20a23d00343eeb,dependency,3,KG_HistoricalIssue,Jira temporal graph HistoricalIssue; source ID...,
3,b72e0d20a23d00343eeb,dependency,4,KG_HistoricalIssue,Jira temporal graph HistoricalIssue; source ID...,
4,b72e0d20a23d00343eeb,dependency,5,JIRA_CHECKPOINT_STATUS,Jira issue MNG-2000: historical status at chec...,
5,041a60f311c0263b395c,dependency,1,ACTIVE_DEPENDENCY,Active Jira dependency: upstream issue ENTESB-...,
6,041a60f311c0263b395c,dependency,2,ACTIVE_DEPENDENCY,Active Jira dependency: upstream issue ENTMQBR...,
7,041a60f311c0263b395c,dependency,3,ACTIVE_DEPENDENCY,Active Jira dependency: upstream issue ENTESB-...,
8,041a60f311c0263b395c,dependency,4,ACTIVE_DEPENDENCY,Active Jira dependency: upstream issue TEIID-5...,
9,041a60f311c0263b395c,dependency,5,JIRA_STATUS_TRANSITION,Observed Jira status transition for issue ENTE...,


## 4. Import labels and calculate metrics **only when complete**

Open `retrieval_relevance_review_top5.csv` in Google Sheets, fill `relevant` with `0` or `1` for **every exported row**, then download/save it as CSV to the same path. Rerun the cell below. If no labels are present, the notebook will report **NOT ESTABLISHED**, not 0%.

For requests returning fewer than five evidence items, report observed-top-K precision and evidence availability separately; do not silently assume missing results are relevant.

In [6]:
judged=pd.read_csv(review_path,keep_default_na=False)
expected=review[['trigger_id','rank','evidence_id']].copy()
assert len(judged)==len(expected),'Review row count changed'
assert judged[['trigger_id','rank','evidence_id']].astype(str).reset_index(drop=True).equals(expected.astype(str).reset_index(drop=True)),'Review keys or row order changed'
labels=judged['relevant'].astype(str).str.strip()
invalid=~labels.isin(['','0','1'])
assert not invalid.any(),f'Invalid labels at rows: {judged.index[invalid].tolist()[:10]}'
complete=labels.isin(['0','1']).all()
summary={'sample_investigations':len(selected),'top_k':TOP_K,'evidence_rows':len(judged),
         'labeled_rows':int(labels.isin(['0','1']).sum()),'complete':bool(complete),
         'precision_at_5':'NOT_ESTABLISHED','recall_at_5':'NOT_ESTABLISHED_NO_COMPLETE_RELEVANCE_UNIVERSE'}
if complete:
    judged['relevant_numeric']=labels.astype(int)
    case=judged.groupby('trigger_id').agg(relevant=('relevant_numeric','sum'),retrieved=('rank','count')).reset_index()
    case['precision_at_observed_k']=case['relevant']/case['retrieved']
    case['precision_at_5_fixed_denominator']=case['relevant']/TOP_K
    case=case.merge(judged[['trigger_id','partition','risk_type','retrieval_status']].drop_duplicates('trigger_id'),on='trigger_id',validate='one_to_one')
    summary.update({'precision_at_5':float(case['precision_at_5_fixed_denominator'].mean()),
        'mean_precision_at_observed_k':float(case['precision_at_observed_k'].mean()),
        'cases_with_fewer_than_5':int(case['retrieved'].lt(TOP_K).sum()),
        'by_risk':case.groupby('risk_type')['precision_at_5_fixed_denominator'].mean().to_dict(),
        'by_partition':case.groupby('partition')['precision_at_5_fixed_denominator'].mean().to_dict()})
    case.to_csv(OUT/'precision_at_5_per_investigation.csv',index=False)
    display(case.head())
else:
    print('Labels incomplete; metrics intentionally withheld.')
(OUT/'retrieval_precision_at_5_summary.json').write_text(json.dumps(summary,indent=2),encoding='utf-8')
print(json.dumps(summary,indent=2))


Labels incomplete; metrics intentionally withheld.
{
  "sample_investigations": 30,
  "top_k": 5,
  "evidence_rows": 144,
  "labeled_rows": 0,
  "complete": false,
  "precision_at_5": "NOT_ESTABLISHED",
  "recall_at_5": "NOT_ESTABLISHED_NO_COMPLETE_RELEVANCE_UNIVERSE"
}


## Interpretation and limitations

- `Precision@5` is the number of relevant items among the first five retrieved slots, divided by five. If fewer than five were returned, unfilled slots count as nonrelevant for this fixed-denominator measure; `precision_at_observed_k` is separately reported.
- Evidence relevance is judged against the **investigation need**, not whether the model cited the evidence.
- This is a purposive stratified 30-case benchmark, so does not establish statistical representativeness or population-wide precision.
- The existing queue may contain fewer than five items for some requests. Keep the denominator convention explicit.
- Recall@5 needs a reasonably complete independently labelled relevant-evidence universe and is **not** established here.
- If judgments are AI-assisted, report them as AI-assisted; for an independent benchmark use blinded human labels.
- Save notebook in `MyDrive/03_Investigation_Explanation/Notebooks/`; outputs in `Retrieval_Evaluation/Results/Precision_at_5_V1/`.